Supreme Court Database (SCDB) oragnized by justice: http://scdb.wustl.edu/data.php

Justice data inc. Segal-Cover score: https://epstein.wustl.edu/justicesdata

# Preparing the Data

In [1]:
#read in SCBD
import pandas as pd
df_SCBD = pd.read_csv("SCDB_2025_01_justiceCentered_Docket.csv.zip")

In [2]:
#read in & clean justice database
df_jscores = pd.read_csv("justicesdata2022.csv")
new_df = df_jscores[(df_jscores["ideo"] != "888. data unavailable") & (df_jscores["ideo"] != "777") & (df_jscores["spaethid"] != "888. no spaeth id number")]#

In [3]:
#select and clean justice features
df_justices_2022 = new_df.drop_duplicates(subset="spaethid")[["spaethid", "ideo", "parnom", "prespart"]]
df_justices_2022 = df_justices_2022.rename(columns={"spaethid": "justice"})
df_justices_2022["justice"] = df_justices_2022["justice"].astype(int)

In [4]:
#add Justice Jackson to justice df

#imputing Justice Kentanji Brown Jackson's score from Justices Elena Kagan and Sonia Sotomayor's scores
jackson_score = (float(df_justices_2022.loc[171]["ideo"]) + float(df_justices_2022.loc[172]["ideo"])) / 2
jackson_row = pd.DataFrame({"justice": [118], "ideo": [jackson_score], "parnom": "1. democrat", "prespart": "1. democrat"})
df_justices = pd.concat([df_justices_2022, jackson_row])
###
df_justices.tail()

,justice,ideo,parnom,prespart
172,114,.73000002,1. democrat,1. democrat
174,115,.11,6. republican,6. republican
175,116,.07,6. republican,6. republican
176,117,.23,6. republican,6. republican
0,118,0.755,1. democrat,1. democrat


In [5]:
#create combined dataframe to use

#merge justices and SCDB dataframce
df_SCBD = df_SCBD.merge(df_justices, on="justice", how="left")

#drop decisions a partisan direction
df_SCBD["ideo"] = df_SCBD["ideo"].astype(float)
df_DD = df_SCBD[df_SCBD["decisionDirection"] != 3].copy()

#map values of 0, 0.5, 1 (Republican, Independent, Democrat) to party features
party_map = {"6. republican": 0, "1. democrat": 1, "5. independent": 0.5}
df_DD["parnom"] = df_DD["parnom"].map(party_map)
df_DD["prespart"] = df_DD["prespart"].map(party_map)

#map values of 0, 1 (Conservative, Liberal) to decision directions
df_DD["decisionDirection"] = df_DD["decisionDirection"].map({2: 1, 1: 0})
df_DD = df_DD.dropna(subset=["decisionDirection"])

# Comparing different measures of ideological lean

In [6]:
#calculate median partisanship (partisanship of middle justice) of each bench
df_median_partisanship = df_DD.groupby("docketId")[["ideo", "parnom", "prespart", "decisionDirection"]].median()

#compare avg bench partisanship for liberal and conservative decisions
df_DD_partisanship = df_median_partisanship.groupby("decisionDirection")[["ideo", "parnom", "prespart"]].mean().T
df_DD_partisanship["difference"] = abs(df_DD_partisanship[0.0] - df_DD_partisanship[1.0])
##
df_DD_partisanship

decisionDirection,0.0,1.0,difference
ideo,0.434793,0.511047,0.076254
parnom,0.340433,0.473517,0.133084
prespart,0.229423,0.360262,0.130839


# Correlation between partisan bench makeup and outcomes

In [7]:
# isolate justice party (most correlative measure of ideology), group by term, and rename columns
df_parnom_DD = df_DD.groupby("term")[["parnom", "decisionDirection"]].mean().reset_index()
df_parnom_DD = df_parnom_DD.rename(columns={"decisionDirection": "Mean score of decisions", "parnom": "Mean score of bench"})

#visualize ideological lean of bench and decisions
import plotly.express as px

fig1 = px.line(
    df_parnom_DD,
    x="term",
    y=["Mean score of decisions", "Mean score of bench"],
    title="Bench Ideology and Decision Direction by Term",
    labels={"term": "Term", "value": "Ideological Score (0 = Conservative, 1 = Liberal)", "variables": "Measure"},
    color_discrete_map={"Mean score of decisions": "#00BA73", "Mean score of bench": "#0071DB"})

fig1.update_traces(line=dict(width=3))

fig1.update_layout(
    width=1500,
    font=dict(size=16),
    title_font=dict(size=20))

fig1.show()

# Ideology-outcome correlation grouping by issue area

In [8]:
#group by issue area and determine those with smallest distance between bench and decision ideology

dists = {}
for issue in df_DD["issueArea"].unique():
    df_issue = df_DD[df_DD["issueArea"] == issue]
    df_parnom_DD2 = df_DD.groupby("term")[["parnom", "decisionDirection"]].mean().reset_index()
    dists[issue] = ((abs(df_parnom_DD2["parnom"] - df_parnom_DD2["decisionDirection"])).mean())
pd.Series(dists).sort_values().head(3)

,0
8.0,0.135461
1.0,0.135461
2.0,0.135461


In [9]:
#graph issue areas with greatest correlation

for issue in [8.0, 1.0, 2.0]:
    df_issue = df_DD[df_DD["issueArea"] == issue]
    df_parnom_DD3 = df_DD.groupby("term")[["parnom", "decisionDirection"]].mean().reset_index()
    fig = px.line(df_parnom_DD3, x="term", y=["decisionDirection", "parnom"],
                  title=f"Issue Area {issue}")
    fig.show()

# Testing ability of ideology to predict outcomes with a logistic regression model

In [10]:
#rename feautures
df_median_partisanship = df_median_partisanship.rename(columns={"ideo": "Segal-Cover ideology score", "parnom": "Party of justice", "prespart": "Party of nominating president"})

In [11]:
#calculate error scores for models trained on each of the three ideology features

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd


features = ["Segal-Cover ideology score", "Party of justice", "Party of nominating president"]
model = LogisticRegression()
errors = {}

for feature in features:
  scores = cross_val_score(
      model,
      X = df_median_partisanship[[feature]],
      y = df_median_partisanship["decisionDirection"],
      cv = 5,
      scoring = "accuracy")
  errors[feature] = scores.mean()

pd.DataFrame(errors, index=["accuracy"]).T


,accuracy
Segal-Cover ideology score,0.572623
Party of justice,0.573658
Party of nominating president,0.513922


In [12]:
#determine random baseline of the model (if it predicted Liberal (1) every time)

random_baseline = df_median_partisanship["decisionDirection"].value_counts(normalize=True).max()
random_baseline

0.5107263831388784

In [13]:
#plot errors of feautres

import plotly.express as px
import plotly.graph_objects as go

fig = px.bar(
    x=list(errors.values()),
    y=list(errors.keys()),
    orientation="h",
    labels={"x": "Accuracy", "y": "Feature"},
    title="Predictive Accuracy by Feature",
    color=list(errors.keys()),
    color_discrete_map={"Party of nominating president": "#89F0BD", "Party of justice": "#00BA73", "Segal-Cover ideology score": "#007549"})

# Add dashed baseline line
fig.add_vline(
    x=random_baseline,
    line_dash="dash",
    line_color="black",
    annotation_text="Random baseline",
    annotation_position="top right")

fig.update_layout(
    height=700,
    width=1600,
    xaxis=dict(range=[0, 0.65]))

fig.show()